# Объединение bias изображений для создания master

Заключительный шаг — объединить отдельные откалиброванные bias изображения в одно объединенное изображение. Это объединенное изображение будет иметь меньше шума, чем отдельные изображения, что минимизирует шум, добавляемый к остальным изображениям при вычитании bias.

Независимо от того, какой путь вы выбрали для калибровки bias (с overscan или без), должна быть папка с именем `reduced`, содержащая откалиброванные bias изображения. Если ее нет, пожалуйста, запустите предыдущий ноутбук перед тем, как продолжить с этим.

In [ ]:
from pathlib import Path
import os

from astropy.nddata import CCDData
from astropy.stats import mad_std

import ccdproc as ccdp
import matplotlib.pyplot as plt
import numpy as np

from convenience_functions import show_image

In [ ]:
# Use custom style for larger fonts and figures
plt.style.use('guide.mplstyle')

## Рекомендуемые настройки для комбинирования изображений

Как обсуждалось в [ноутбуке о комбинировании изображений](01-06-Image-combination.ipynb), рекомендация заключается в том, чтобы комбинировать путем усреднения отдельных изображений, но с sigma clipping для удаления экстремальных значений.

[ccdproc](https://ccdproc.readthedocs.org) предоставляет два способа комбинирования:

+ Объектно-ориентированный интерфейс, построенный вокруг объекта `Combiner`, описанный в [документации ccdproc по комбинированию изображений](https://ccdproc.readthedocs.io/en/latest/image_combination.html).
+ Функцию [`combine`](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.combine.html#ccdproc.combine), которую мы будем использовать здесь, потому что функция позволяет указать максимальный объем памяти, который должен использоваться во время комбинирования. Эта функция может быть необходима в зависимости от того, сколько изображений нужно объединить, насколько они велики и сколько памяти есть у вашего компьютера.

*ПРИМЕЧАНИЕ: Если используется версия ccdproc ниже 2.0, установите ограничение памяти в 2-3 раза ниже, чем вы хотите, чтобы было максимальное потребление памяти.*

## Пример 1: Криогенно охлаждаемая камера

Остальная часть этого раздела предполагает, что откалиброванные bias изображения находятся в папке `example1-reduced`, которая создается в предыдущем ноутбуке.

In [ ]:
calibrated_path = Path('example1-reduced')
reduced_images = ccdp.ImageFileCollection(calibrated_path)

Приведенный ниже код:

+ выбирает откалиброванные bias изображения,
+ объединяет их с помощью функции `combine`,
+ добавляет ключевое слово `COMBINED` в заголовок, чтобы на последующих этапах калибровки можно было легко определить, какой bias использовать, и
+ записывает файл.

In [ ]:
calibrated_biases = reduced_images.files_filtered(imagetyp='bias', include_path=True)

combined_bias = ccdp.combine(calibrated_biases,
                             method='average',
                             sigma_clip=True, sigma_clip_low_thresh=5, sigma_clip_high_thresh=5,
                             sigma_clip_func=np.ma.median, sigma_clip_dev_func=mad_std,
                             mem_limit=350e6
                            )

combined_bias.meta['combined'] = True

combined_bias.write(calibrated_path / 'combined_bias.fit')

### Результат для Примера 1

Ниже показаны одиночное откалиброванное изображение и объединенное изображение. В bias есть значительная двумерная структура, которую нельзя легко удалить, вычитая только overscan на следующих этапах редукции изображений. Получение bias изображений занимает мало времени, и это приведет к более высокому качеству научных изображений.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

show_image(CCDData.read(calibrated_biases[0]).data, cmap='gray', ax=ax1, fig=fig, percl=90)
ax1.set_title('Single calibrated bias')
show_image(combined_bias.data, cmap='gray', ax=ax2, fig=fig, percl=90)
ax2.set_title('{} bias images combined'.format(len(calibrated_biases)))

## Пример 2: Термоэлектрически охлаждаемая камера

Процесс объединения изображений точно такой же, как в примере 1. Единственное отличие — это каталог, содержащий откалиброванные bias кадры.

In [ ]:
calibrated_path = Path('example2-reduced')
reduced_images = ccdp.ImageFileCollection(calibrated_path)

Приведенный ниже код:

+ выбирает откалиброванные bias изображения,
+ объединяет их с помощью функции `combine`,
+ добавляет ключевое слово `COMBINED` в заголовок, чтобы на последующих этапах калибровки можно было легко определить, какой bias использовать, и
+ записывает файл.

In [ ]:
calibrated_biases = reduced_images.files_filtered(imagetyp='bias', include_path=True)

combined_bias = ccdp.combine(calibrated_biases,
                             method='average',
                             sigma_clip=True, sigma_clip_low_thresh=5, sigma_clip_high_thresh=5,
                             sigma_clip_func=np.ma.median, signma_clip_dev_func=mad_std,
                             mem_limit=350e6
                            )

combined_bias.meta['combined'] = True

combined_bias.write(calibrated_path / 'combined_bias.fit')

### Результат для Примера 2

Разница между одиночным откалиброванным bias изображением и объединенным bias изображением в этом случае гораздо более заметна.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

show_image(CCDData.read(calibrated_biases[0]).data, cmap='gray', ax=ax1, fig=fig)
ax1.set_title('Single calibrated bias')
show_image(combined_bias.data, cmap='gray', ax=ax2, fig=fig)
ax2.set_title('{} bias images combined'.format(len(calibrated_biases)))